# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = 'llama3.2:3b'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:24<00:00,  8.16s/it]


In [3]:
len(deals)

30

In [5]:
print(deals[10].describe())

Title: Yxk Zero1 Pro 4-Bay NAS for $240 + free shipping
Details: Home users needing centralized storage and backups across multiple devices will get the most out of this 4-bay setup. Apply coupon code "SBNLZ39O" for a savings of $160. Buy Now at Amazon
Features: Supports 144TB total storage across six drive slots Dual 2.5GbE ports for high-speed file transfers Direct 4K HDMI output for home theater streaming AI-powered photo organization and auto-tagging Full Docker support for custom application hosting
URL: https://www.dealnews.com/Yxk-Zero1-Pro-4-Bay-NAS-for-240-free-shipping/22134350.html?iref=rss-c39


### We are going to ask GPT-5-mini to summarize deals and identify their price

In [6]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [7]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [8]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Forging Mount No Stud TV Wall Mount for 26" to 100" TVs for $20 w/ Prime + free shipping
Details: This no-stud mount is worth a look if your wall studs don't line up where you need the TV, since it holds 26"-100" screens without requiring you to hit them. Apply coupon code "2TDHPXFV" for a savings of $30. This deal is for Prime members only. Buy Now at Amazon
Features: 
URL: ht

In [16]:
response = openai.chat.completions.parse(model="qwen3.5:2b", messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='A high-quality wireless entertainment system for home use that includes speakers delivering stereo sound experience, features Dolby Atmos surround sound support, DTS:X compatibility, WOWCAST wireless connectivity, and a 25% increase in power output. Includes a wireless subwoofer setup with no need for separate receivers to connect.', price=382.0, url='https://www.dealnews.com/LG-5-1-3-Channel-Dolby-Atmos-Soundbar-w-Speakers-for-382-free-shipping/22146391.html?iref=rss-c142')])

In [17]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


A high-quality wireless entertainment system for home use that includes speakers delivering stereo sound experience, features Dolby Atmos surround sound support, DTS:X compatibility, WOWCAST wireless connectivity, and a 25% increase in power output. Includes a wireless subwoofer setup with no need for separate receivers to connect.
382.0
https://www.dealnews.com/LG-5-1-3-Channel-Dolby-Atmos-Soundbar-w-Speakers-for-382-free-shipping/22146391.html?iref=rss-c142



In [18]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [25]:
import importlib
import agents.scanner_agent

importlib.reload(agents.scanner_agent)
from agents.scanner_agent import ScannerAgent

In [26]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 4 selected deals with price>0 from OpenAI


In [29]:
print(result)

deals=[Deal(product_description='This no-stud mount is worth a look if your wall studs don\'t line up where you need the TV, since it holds 26" to 100" screens without requiring you to hit them. The mount is fully adjustable and includes a 34" VESA pattern, so you can position it to fit your TV perfectly. The build is also very sturdy and can hold up to 80 lbs. Apply coupon code "2TDHPXFV" for a savings of $30. Buy Now at Amazon', price=20.0, url='https://www.dealnews.com/Forging-Mount-No-Stud-TV-Wall-Mount-for-26-to-100-TVs-for-20-w-Prime-free-shipping/22124118.html?iref=rss-c142'), Deal(product_description='This suits businesses needing a large-format display for lobbies, retail spaces, or wayfinding without mounting it to a wall, since the floor stand setup makes repositioning simple. The display has a durable design and includes Android OS with cloud-based content management, Split-screen functionality for multi-content display, Flexible USB, HDMI, and 5G Wi-Fi connectivity, 600-ni

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [37]:
load_dotenv(override=True)

True

In [38]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [39]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [33]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [41]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [45]:
import importlib
import agents.messaging_agent

importlib.reload(agents.messaging_agent)

from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [46]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
13:23:02 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= llama3.2:3b; provider = ollama
INFO:LiteLLM:
LiteLLM completion() model= llama3.2:3b; provider = ollama
13:23:05 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
